# Adversarial Frozen-Opponent MAPPO

Small launcher for the experimental two-team JAX MAPPO lane. It runs the adversarial workflow config, optionally actor-warm-starting from a compatible cooperative checkpoint.

In [ ]:
from pathlib import Path
import os
import sys

# Set these before importing JAX in this kernel.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.35")
if "jax" in sys.modules:
    print("Restart the kernel before rerunning training cells so JAX sees the memory settings.")

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
PACKAGE_ROOT = PROJECT_ROOT / "src" / "ant_byte_env"
sys.path.insert(0, str(PACKAGE_ROOT.parent))

CONFIG = PROJECT_ROOT / "experiments" / "adversarial_frozen_opponent_shared_writes_8ants.json"
SOURCE_EXPERIMENT_CONFIG = PROJECT_ROOT / "experiments" / "exploration_to_forage_full_layout_8ants_half_food_50x50_shared_writes.json"
RUN_ROOT = PROJECT_ROOT / "runs" / "notebooks" / "adversarial_frozen_opponent"
RUN_ROOT.mkdir(parents=True, exist_ok=True)

print(PROJECT_ROOT)
print(CONFIG)
print(SOURCE_EXPERIMENT_CONFIG)
print(RUN_ROOT)

In [ ]:
import importlib
import json

import jax
from IPython.display import Video, display
from tqdm.auto import tqdm

from ant_byte_env import AntByteForagingEnv, notebook_workflows as workflows
from ant_byte_env.experiments import config_args_to_argv, load_experiment_config
from ant_byte_env.training.jax_mappo.adversarial.checkpointing import evaluate_checkpoint_matrix
from ant_byte_env.training.jax_mappo.adversarial.rendering import render_adversarial_rollout
from ant_byte_env.training.jax_mappo.adversarial import runner as adversarial_runner

workflows = importlib.reload(workflows)
adversarial_runner = importlib.reload(adversarial_runner)

resource_status = workflows.configure_jax_notebook_runtime(memory_fraction="0.35")
workflows.assert_notebook_resources_available(resource_status)
print(f"JAX device: {jax.devices()[0]}")
resource_status

## Training Settings

In [ ]:
RUN_TRAINING = True
EVALUATION_EPISODES = 1
EVALUATION_MAX_STEPS = 500
EVALUATION_PROGRESS_INTERVAL = 25
RENDER_SECONDS = 60
RENDER_FPS = AntByteForagingEnv.metadata["render_fps"]
RENDER_MAX_FRAMES = RENDER_SECONDS * RENDER_FPS
RUN_NAME = "warmstart_shared_writes_8ants_sparse_adversarial_interior_midpoint_curriculum_10k_cpu_fast"
CURRICULUM_ROOT = RUN_ROOT / RUN_NAME
CURRICULUM_ROOT.mkdir(parents=True, exist_ok=True)

# Fresh adversarial run: start stage 1 from the cooperative actor checkpoint,
# not from a previously trained adversarial checkpoint.
CURRICULUM_START_CHECKPOINT = None

# Controlled diagnostic layout: hubs are sampled from the interior, with food near their midpoint.
DIAGNOSTIC_PLACEMENT_OVERRIDES = [
    "--food-count", "125",
    "--food-sources", "4",
    "--layout-margin", "6",
    "--hub-center-window-size", "0",
    "--hub-pair-distance-min", "12",
    "--hub-pair-distance-max", "20",
    "--food-midpoint-window-size", "8",
]

# Fixed-N curriculum: run each stage for this many PPO updates, then make food scarcer,
# widen the hub-distance range, fade out the central hub window, and keep food in a midpoint contest zone.
# No reward shaping is added here; training reward stays own deliveries minus opponent deliveries.
CURRICULUM_UPDATES_PER_STAGE = 1000
FOOD_CURRICULUM = [
    {
        "label": "food1000_src16_dist06_10_mid08_team0",
        "food_count": 1000,
        "food_sources": 16,
        "layout_margin": 0,
        "hub_center_window_size": 18,
        "hub_pair_distance_min": 6,
        "hub_pair_distance_max": 10,
        "food_midpoint_window_size": 8,
        "learner_team": 0,
        "ent_coef": 0.02,
        "learning_rate": 2.5e-4,
    },
    {
        "label": "food875_src14_dist06_12_mid08_team1",
        "food_count": 875,
        "food_sources": 14,
        "layout_margin": 0,
        "hub_center_window_size": 24,
        "hub_pair_distance_min": 6,
        "hub_pair_distance_max": 12,
        "food_midpoint_window_size": 8,
        "learner_team": 1,
        "ent_coef": 0.02,
        "learning_rate": 2.5e-4,
    },
    {
        "label": "food750_src12_dist08_14_mid10_team0",
        "food_count": 750,
        "food_sources": 12,
        "layout_margin": 0,
        "hub_center_window_size": 30,
        "hub_pair_distance_min": 8,
        "hub_pair_distance_max": 14,
        "food_midpoint_window_size": 10,
        "learner_team": 0,
        "ent_coef": 0.015,
        "learning_rate": 2.5e-4,
    },
    {
        "label": "food625_src10_dist08_16_mid10_team1",
        "food_count": 625,
        "food_sources": 10,
        "layout_margin": 0,
        "hub_center_window_size": 36,
        "hub_pair_distance_min": 8,
        "hub_pair_distance_max": 16,
        "food_midpoint_window_size": 10,
        "learner_team": 1,
        "ent_coef": 0.015,
        "learning_rate": 2.5e-4,
    },
    {
        "label": "food500_src8_dist10_18_mid12_team0",
        "food_count": 500,
        "food_sources": 8,
        "layout_margin": 4,
        "hub_center_window_size": 42,
        "hub_pair_distance_min": 10,
        "hub_pair_distance_max": 18,
        "food_midpoint_window_size": 12,
        "learner_team": 0,
        "ent_coef": 0.01,
        "learning_rate": 2.0e-4,
    },
    {
        "label": "food375_src6_dist12_20_mid12_team1",
        "food_count": 375,
        "food_sources": 6,
        "layout_margin": 6,
        "hub_center_window_size": 0,
        "hub_pair_distance_min": 12,
        "hub_pair_distance_max": 20,
        "food_midpoint_window_size": 12,
        "learner_team": 1,
        "ent_coef": 0.01,
        "learning_rate": 2.0e-4,
    },
    {
        "label": "food312_src5_dist14_24_mid14_team0",
        "food_count": 312,
        "food_sources": 5,
        "layout_margin": 6,
        "hub_center_window_size": 0,
        "hub_pair_distance_min": 14,
        "hub_pair_distance_max": 24,
        "food_midpoint_window_size": 14,
        "learner_team": 0,
        "ent_coef": 0.007,
        "learning_rate": 2.0e-4,
    },
    {
        "label": "food250_src4_dist16_28_mid14_team1",
        "food_count": 250,
        "food_sources": 4,
        "layout_margin": 6,
        "hub_center_window_size": 0,
        "hub_pair_distance_min": 16,
        "hub_pair_distance_max": 28,
        "food_midpoint_window_size": 14,
        "learner_team": 1,
        "ent_coef": 0.007,
        "learning_rate": 2.0e-4,
    },
    {
        "label": "food188_src3_dist18_32_mid16_team0",
        "food_count": 188,
        "food_sources": 3,
        "layout_margin": 6,
        "hub_center_window_size": 0,
        "hub_pair_distance_min": 18,
        "hub_pair_distance_max": 32,
        "food_midpoint_window_size": 16,
        "learner_team": 0,
        "ent_coef": 0.005,
        "learning_rate": 1.5e-4,
    },
    {
        "label": "target_125food_2src_dist20_36_mid16_team1",
        "food_count": 125,
        "food_sources": 2,
        "layout_margin": 6,
        "hub_center_window_size": 0,
        "hub_pair_distance_min": 20,
        "hub_pair_distance_max": 36,
        "food_midpoint_window_size": 16,
        "learner_team": 1,
        "ent_coef": 0.005,
        "learning_rate": 1.5e-4,
    },
]

STAGE_ARG_KEYS = {
    "food_count": "food-count",
    "food_sources": "food-sources",
    "layout_margin": "layout-margin",
    "hub_center_window_size": "hub-center-window-size",
    "hub_pair_distance_min": "hub-pair-distance-min",
    "hub_pair_distance_max": "hub-pair-distance-max",
    "food_midpoint_window_size": "food-midpoint-window-size",
    "learner_team": "learner-team",
    "ent_coef": "ent-coef",
    "learning_rate": "learning-rate",
}

spec = load_experiment_config(CONFIG)
source_spec = load_experiment_config(SOURCE_EXPERIMENT_CONFIG)
COOPERATIVE_CHECKPOINT = (PROJECT_ROOT / spec.args["learner_load_model"]).resolve()
if not COOPERATIVE_CHECKPOINT.exists():
    raise FileNotFoundError(COOPERATIVE_CHECKPOINT)
if CURRICULUM_START_CHECKPOINT is not None and not CURRICULUM_START_CHECKPOINT.exists():
    raise FileNotFoundError(CURRICULUM_START_CHECKPOINT)

BASE_TRAINING_ARGS = dict(spec.args)
BASE_TRAINING_ARGS.pop("learner_load_model", None)
BASE_TRAINING_ARGV = config_args_to_argv(BASE_TRAINING_ARGS)
steps_per_update = int(spec.args["num_envs"]) * int(spec.args["num_steps"])
stage_timesteps = int(CURRICULUM_UPDATES_PER_STAGE * steps_per_update)
CURRICULUM_SUMMARY_PATH = CURRICULUM_ROOT / "curriculum_summary.json"


def stage_run_dir(stage_index):
    stage = FOOD_CURRICULUM[stage_index]
    return CURRICULUM_ROOT / f"stage_{stage_index + 1:02d}_{stage['label']}"


def stage_checkpoint(stage_index):
    return stage_run_dir(stage_index) / "checkpoints" / "model.pkl"


def build_stage_overrides(stage_index, resume_checkpoint=None):
    stage = FOOD_CURRICULUM[stage_index]
    overrides = [
        "--run-dir",
        str(stage_run_dir(stage_index)),
        "--total-timesteps",
        str(stage_timesteps),
        "--eval-episodes",
        "0",
    ]
    for key, option_name in STAGE_ARG_KEYS.items():
        overrides.extend([f"--{option_name}", str(stage[key])])
    if resume_checkpoint is None:
        overrides.extend(["--learner-load-model", str(COOPERATIVE_CHECKPOINT)])
    else:
        overrides.extend([
            "--resume-model",
            str(resume_checkpoint),
            "--opponent-load-model",
            str(COOPERATIVE_CHECKPOINT),
        ])
    return overrides


def build_stage_argv(stage_index, resume_checkpoint=None):
    return [*BASE_TRAINING_ARGV, *build_stage_overrides(stage_index, resume_checkpoint)]


def build_diagnostic_argv(argv):
    return [*argv, *DIAGNOSTIC_PLACEMENT_OVERRIDES]


FINAL_STAGE_INDEX = len(FOOD_CURRICULUM) - 1
CHECKPOINT_PATH = stage_checkpoint(FINAL_STAGE_INDEX)
training_argv = build_stage_argv(0, CURRICULUM_START_CHECKPOINT)
diagnostic_argv = build_diagnostic_argv(training_argv)
ACTIVE_CHECKPOINT_PATH = CHECKPOINT_PATH if CHECKPOINT_PATH.exists() else None

json.dumps({
    "experiment": spec.name,
    "source_experiment": source_spec.name,
    "warm_start_checkpoint": str(COOPERATIVE_CHECKPOINT),
    "curriculum_start_checkpoint": str(CURRICULUM_START_CHECKPOINT) if CURRICULUM_START_CHECKPOINT else None,
    "curriculum_root": str(CURRICULUM_ROOT),
    "updates_per_stage": CURRICULUM_UPDATES_PER_STAGE,
    "stage_timesteps": stage_timesteps,
    "total_curriculum_updates": CURRICULUM_UPDATES_PER_STAGE * len(FOOD_CURRICULUM),
    "total_curriculum_timesteps": stage_timesteps * len(FOOD_CURRICULUM),
    "render_seconds": RENDER_SECONDS,
    "render_max_frames": RENDER_MAX_FRAMES,
    "diagnostic_placement_overrides": DIAGNOSTIC_PLACEMENT_OVERRIDES,
    "final_checkpoint": str(CHECKPOINT_PATH),
    "active_checkpoint": str(ACTIVE_CHECKPOINT_PATH) if ACTIVE_CHECKPOINT_PATH else None,
    "food_curriculum": FOOD_CURRICULUM,
}, indent=2)


In [ ]:
from ant_byte_env.cli import main as ant_byte_main

ant_byte_main([
    "train",
    "jax",
    "--config",
    str(CONFIG),
    "--dry-run",
    "--",
    *build_stage_overrides(0, CURRICULUM_START_CHECKPOINT),
])


## Run

In [ ]:
total_curriculum_updates = CURRICULUM_UPDATES_PER_STAGE * len(FOOD_CURRICULUM)
progress_rows = []
stage_results = []
progress_state = {"completed_updates": 0}


def make_stage_progress_callback(stage_index, stage):
    def record_progress(update_index, total_update_count, train_metrics):
        completed_updates = stage_index * CURRICULUM_UPDATES_PER_STAGE + int(update_index)
        update_delta = max(0, completed_updates - progress_state["completed_updates"])
        progress_state["completed_updates"] = completed_updates
        if progress_bar is not None and update_delta:
            progress_bar.update(update_delta)
            progress_bar.set_postfix(
                stage=f"{stage_index + 1}/{len(FOOD_CURRICULUM)}",
                food=f"{stage['food_count']}/{stage['food_sources']}",
                hub=f"{stage['hub_center_window_size']}",
                margin=f"{stage['layout_margin']}",
                dist=f"{stage['hub_pair_distance_min']}-{stage['hub_pair_distance_max']}",
                mid=f"{stage['food_midpoint_window_size']}",
                team=f"{stage['learner_team']}",
                ent=f"{stage['ent_coef']:.3f}",
                loss=f"{train_metrics['loss']:.3f}",
                ret=f"{train_metrics['episode_return']:.2f}",
                diff=f"{train_metrics['delivery_event_difference']:.0f}",
            )
        progress_rows.append({
            "stage_index": int(stage_index + 1),
            "stage_label": stage["label"],
            "update": int(update_index),
            "total_updates": int(total_update_count),
            **train_metrics,
        })

    return record_progress


if RUN_TRAINING:
    previous_checkpoint = CURRICULUM_START_CHECKPOINT
    progress_bar = tqdm(
        total=total_curriculum_updates,
        desc="adversarial hub/food curriculum",
        bar_format="{desc}: {n_fmt}/{total_fmt} updates |{bar}| {elapsed}<{remaining} {postfix}",
        leave=True,
    )
    try:
        for stage_index, stage in enumerate(FOOD_CURRICULUM):
            current_argv = build_stage_argv(stage_index, previous_checkpoint)
            metrics = adversarial_runner.main(
                current_argv,
                progress_callback=make_stage_progress_callback(stage_index, stage),
            )
            current_checkpoint = stage_checkpoint(stage_index)
            if not current_checkpoint.exists():
                raise FileNotFoundError(current_checkpoint)
            stage_results.append({
                "stage_index": stage_index + 1,
                "stage": stage,
                "checkpoint": str(current_checkpoint),
                "resume_checkpoint": str(previous_checkpoint) if previous_checkpoint else None,
                "metrics": metrics,
                "argv": current_argv,
            })
            previous_checkpoint = current_checkpoint
    finally:
        progress_bar.close()

    training_argv = stage_results[-1]["argv"]
    diagnostic_argv = build_diagnostic_argv(training_argv)
    metrics = stage_results[-1]["metrics"]
    ACTIVE_CHECKPOINT_PATH = CHECKPOINT_PATH if CHECKPOINT_PATH.exists() else None
    CURRICULUM_SUMMARY_PATH.write_text(
        json.dumps({
            "curriculum_root": str(CURRICULUM_ROOT),
            "curriculum_start_checkpoint": str(CURRICULUM_START_CHECKPOINT) if CURRICULUM_START_CHECKPOINT else None,
            "updates_per_stage": CURRICULUM_UPDATES_PER_STAGE,
            "stage_timesteps": stage_timesteps,
            "total_curriculum_updates": total_curriculum_updates,
            "total_curriculum_timesteps": stage_timesteps * len(FOOD_CURRICULUM),
            "final_checkpoint": str(CHECKPOINT_PATH),
            "diagnostic_placement_overrides": DIAGNOSTIC_PLACEMENT_OVERRIDES,
            "stages": stage_results,
        }, indent=2),
        encoding="utf-8",
    )
else:
    progress_bar = None
    metrics = {"status": "skipped", "set_RUN_TRAINING": True}

{"metrics": metrics, "stage_results": stage_results, "progress_rows": progress_rows[-5:]}


In [ ]:
if CURRICULUM_SUMMARY_PATH.exists():
    curriculum_summary = json.loads(CURRICULUM_SUMMARY_PATH.read_text(encoding="utf-8"))
    compact_summary = {
        "final_checkpoint": curriculum_summary["final_checkpoint"],
        "updates_per_stage": curriculum_summary["updates_per_stage"],
        "stages": [
            {
                "stage_index": stage["stage_index"],
                "stage": stage["stage"],
                "checkpoint": stage["checkpoint"],
                "metrics": stage["metrics"],
            }
            for stage in curriculum_summary["stages"]
        ],
    }
    print(json.dumps(compact_summary, indent=2, sort_keys=True))
else:
    print(f"No curriculum summary yet: {CURRICULUM_SUMMARY_PATH}")

for stage_index, stage in enumerate(FOOD_CURRICULUM):
    metrics_path = stage_run_dir(stage_index) / "metrics.jsonl"
    if metrics_path.exists():
        print(metrics_path)
        print(metrics_path.read_text(encoding="utf-8").splitlines()[-1])


## Evaluation

In [ ]:
if ACTIVE_CHECKPOINT_PATH is not None and ACTIVE_CHECKPOINT_PATH.exists():
    matchup_order = [
        "frozen_vs_frozen",
        "learner_vs_frozen",
        "frozen_vs_learner",
        "random_vs_frozen",
        "learner_vs_random",
    ]
    matchup_offsets = {name: index for index, name in enumerate(matchup_order)}
    evaluation_total_steps = len(matchup_order) * EVALUATION_EPISODES * EVALUATION_MAX_STEPS
    evaluation_progress_rows = []
    evaluation_progress_state = {"completed_steps": 0}
    evaluation_bar = tqdm(
        total=evaluation_total_steps,
        desc="adversarial eval",
        bar_format="{desc}: {n_fmt}/{total_fmt} steps |{bar}| {elapsed}<{remaining} {postfix}",
        leave=True,
    )

    def record_evaluation_progress(matchup_name, episode_index, total_episodes, eval_metrics_row):
        matchup_offset = matchup_offsets[matchup_name]
        completed_steps = int(
            (
                matchup_offset * total_episodes * EVALUATION_MAX_STEPS
                + (episode_index - 1) * EVALUATION_MAX_STEPS
                + eval_metrics_row["step"]
            )
        )
        step_delta = max(0, completed_steps - evaluation_progress_state["completed_steps"])
        evaluation_progress_state["completed_steps"] = completed_steps
        if step_delta:
            evaluation_bar.update(step_delta)
        evaluation_bar.set_postfix(
            matchup=matchup_name,
            episode=f"{episode_index}/{total_episodes}",
            step=f"{int(eval_metrics_row['step'])}/{EVALUATION_MAX_STEPS}",
            event=eval_metrics_row["event"],
        )
        evaluation_progress_rows.append({
            "matchup": matchup_name,
            "episode": int(episode_index),
            "total_episodes": int(total_episodes),
            **eval_metrics_row,
        })

    try:
        evaluation_metrics = evaluate_checkpoint_matrix(
            ACTIVE_CHECKPOINT_PATH,
            argv=diagnostic_argv,
            eval_episodes=EVALUATION_EPISODES,
            eval_max_steps=EVALUATION_MAX_STEPS,
            progress_callback=record_evaluation_progress,
            progress_step_interval=EVALUATION_PROGRESS_INTERVAL,
        )
    finally:
        evaluation_bar.close()
else:
    evaluation_progress_rows = []
    evaluation_metrics = {"status": "missing_checkpoint", "checkpoint": str(ACTIVE_CHECKPOINT_PATH) if ACTIVE_CHECKPOINT_PATH else None}

{"evaluation_metrics": evaluation_metrics, "evaluation_progress_rows": evaluation_progress_rows}


## Render

In [ ]:
if ACTIVE_CHECKPOINT_PATH is not None and ACTIVE_CHECKPOINT_PATH.exists():
    rollout_video = render_adversarial_rollout(
        ACTIVE_CHECKPOINT_PATH,
        CURRICULUM_ROOT / "media" / "adversarial_rollout.mp4",
        argv=diagnostic_argv,
        max_frames=RENDER_MAX_FRAMES,
        tile_size=22,
        action_mode=spec.args["eval_action_mode"],
    )
    display(Video(str(rollout_video), embed=True))
else:
    rollout_video = None
    print(f"No checkpoint yet: {CHECKPOINT_PATH}; active fallback: {ACTIVE_CHECKPOINT_PATH}")

rollout_video
